# Test Suite Minimization Exercise

In this exercise, we'll practice **test suite minimization** — given a (deliberately redundant) test suite for `calculate_shipping_fee`, find a smaller subset that **preserves the original mutation score**.

We'll:
1. Build a large category-grid test suite for `calculate_shipping_fee`.
2. Generate mutants with **mutmut** (see [the mutation-testing exercise](../../07_mutation_testing/exercises/01_python_mutation_testing_exercise2.ipynb)).
3. Build a **test × mutant kill matrix** in-process.
4. Apply **greedy set-cover** to minimize the suite while keeping every originally-killed mutant killed.

Run this notebook with the working directory set to `modules/09_regression_testing/exercises/`.

## Software under Test (SuT)

We reuse the `calculate_shipping_fee` function. Its source is in `calculate_shipping_fee.py` (same logic as the mutation-testing exercise) — we need it as a real `.py` file so `mutmut` can mutate it on disk.

In [1]:
from calculate_shipping_fee import calculate_shipping_fee

# Sanity check
print(calculate_shipping_fee(30000, 10.0, 20, False, "NONE", "NONE"))   # expect 6000
print(calculate_shipping_fee(30000, 10.0, 20, False, "WOW", "NONE"))    # expect 3000
print(calculate_shipping_fee(50000, 3.0, 5, False, "NONE", "NEW_USER")) # expect 0

6000
3000
0


## Build a Large Test Suite

We generate a **category-grid** test suite by combining boundary and representative values for each input. Each test is an `(input_tuple, expected_output)` pair, where the expected output is the result of running the **original** `calculate_shipping_fee` (a *golden* / *characterization* suite).

Why a redundant suite? Many of these tests overlap in what they exercise — that's exactly the redundancy we want to **minimize away**.

In [2]:
from itertools import product

ORDER_TOTAL_VALUES = [10000, 39999, 45000, 50000]     # below / boundary / above the 40000 threshold (+ a value in [40000, 50000) for Ex2 compatibility)
WEIGHT_VALUES      = [3.0, 5.0, 20.0, 25.0]           # all 3 weight tiers + boundaries
DISTANCE_VALUES    = [5, 10, 50, 100]                 # all 3 distance tiers + boundaries
IS_ISLAND_VALUES   = [True, False]
MEMBERSHIP_VALUES  = ["WOW", "NONE"]
COUPON_VALUES      = ["NEW_USER", "NONE"]

def build_test_suite():
    suite = []
    for inp in product(
        ORDER_TOTAL_VALUES, WEIGHT_VALUES, DISTANCE_VALUES,
        IS_ISLAND_VALUES, MEMBERSHIP_VALUES, COUPON_VALUES,
    ):
        expected = calculate_shipping_fee(*inp)
        suite.append((inp, expected))
    return suite

test_suite = build_test_suite()
print(f"Generated {len(test_suite)} tests")
print(f"First 3 tests: {test_suite[:3]}")

Generated 512 tests
First 3 tests: [((10000, 3.0, 5, True, 'WOW', 'NEW_USER'), 1500), ((10000, 3.0, 5, True, 'WOW', 'NONE'), 3500), ((10000, 3.0, 5, True, 'NONE', 'NEW_USER'), 5000)]


## Generate Mutants with `mutmut`

We let `mutmut` generate all mutants of `calculate_shipping_fee.py`. We pair it with a **dummy passing test** in `_mutmut_baseline/` so that mutmut's own pass/fail bookkeeping shows every mutant as `Survived` — we'll compute kills ourselves below.

In [4]:
!rm -rf .mutmut-cache
!mutmut run --paths-to-mutate calculate_shipping_fee.py --tests-dir _mutmut_baseline


- Mutation testing starting -

These are the steps:
1. A full test suite run will be made to make sure we
   can run the tests successfully and we know how long
   it takes (to detect infinite loops for example)
2. Mutants will be generated and checked

Results are stored in .mutmut-cache.
Print found mutants with `mutmut results`.

Legend for output:
🎉 Killed mutants.   The goal is for everything to end up in this bucket.
⏰ Timeout.          Test suite took 10 times as long as the baseline so were killed.
🤔 Suspicious.       Tests took a long time, but not long enough to be fatal.
🙁 Survived.         This means your tests need to be expanded.
🔇 Skipped.          Skipped.

mutmut cache is out of date, clearing it...
1. Running tests without mutations
⠏ Running...Done

2. Checking mutants
⠦ 47/47  🎉 0  ⏰ 0  🤔 0  🙁 47  🔇 0


In [5]:
!mutmut results

To apply a mutant on disk:
    mutmut apply <id>

To show a mutant:
    mutmut show <id>


Survived 🙁 (47)

---- calculate_shipping_fee.py (47) ----

1-47


### Collect all mutant IDs

With the dummy baseline, every mutant should be in the `survived` bucket — that's our complete mutant set.

In [6]:
import subprocess

def get_all_mutant_ids():
    out = subprocess.run(
        ["mutmut", "result-ids", "survived"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    return [int(x) for x in out.split()] if out else []

mutant_ids = get_all_mutant_ids()
print(f"Total mutants: {len(mutant_ids)}")
print(f"IDs: {mutant_ids}")

Total mutants: 47
IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47]


## Extract Each Mutant as a Callable

`mutmut apply <id>` rewrites the source file with that mutation applied. To get a *callable* mutant we:

1. Snapshot the original source.
2. For each mutant id: apply, read the mutated source, **restore** the original, then `exec()` the mutated source into a fresh namespace and grab its `calculate_shipping_fee`.

If a mutant fails to compile/exec (e.g., a mutation that creates a syntax error), we treat it as **automatically killed** — any test would detect it.

In [7]:
SOURCE_PATH = "calculate_shipping_fee.py"

def load_mutants(source_path, mutant_ids):
    with open(source_path) as f:
        original_src = f.read()

    mutants = {}
    try:
        for mid in mutant_ids:
            subprocess.run(
                ["mutmut", "apply", str(mid)],
                capture_output=True, text=True, check=True,
            )
            with open(source_path) as f:
                mutated_src = f.read()
            with open(source_path, "w") as f:
                f.write(original_src)  # restore immediately

            ns = {}
            try:
                exec(compile(mutated_src, f"<mutant_{mid}>", "exec"), ns)
                mutants[mid] = ns.get("calculate_shipping_fee")
            except Exception:
                mutants[mid] = None  # broken mutant -> auto-killed
    finally:
        with open(source_path, "w") as f:
            f.write(original_src)
    return mutants

mutants = load_mutants(SOURCE_PATH, mutant_ids)
print(f"Loaded {len(mutants)} mutants "
      f"({sum(1 for m in mutants.values() if m is not None)} executable, "
      f"{sum(1 for m in mutants.values() if m is None)} broken)")

Loaded 47 mutants (47 executable, 0 broken)


## Build the Test × Mutant Kill Matrix

A test **kills** a mutant if the mutant's output differs from the expected output (or the mutant raises). We store, for each test, the set of mutant IDs it kills.

```
kill_matrix[test_id] = { mutant_id, mutant_id, ... }
```

In [12]:
def build_kill_matrix(test_suite, mutants):
    """Build the test x mutant kill matrix.

    For each test in `test_suite`, determine which mutants it "kills".
    A mutant is KILLED by a test when running that mutant on the test's
    input produces an output different from `expected` (the golden output
    of the original function), OR when it raises an exception.

    Why we want this matrix:
      - rows = tests, columns = mutants, cell = killed? (yes/no)
      - tests that kill many mutants are valuable
      - tests whose killed-set is a subset of another test's are redundant
      - greedy minimization picks the smallest set of rows whose union of
        killed mutants equals the original suite's union (set-cover).

    Returns:
        dict mapping test_id (int) -> set of killed mutant_ids (set[int]).
    """
    kill_matrix = {}
    for tid, (inp, expected) in enumerate(test_suite):
        killed = set()
        for mid, mut_func in mutants.items():
            # `mut_func is None` means the mutant's source failed to compile
            # or exec (e.g., a mutation that produces invalid Python). Such a
            # mutant is trivially detected by any test, so we record it as
            # killed by every test and skip running it.
            if mut_func is None:
                killed.add(mid)
                continue

            # Run the mutant on this test's input. If it raises, its behavior
            # already differs from the original (which doesn't raise on these
            # inputs by construction of `expected`) -> killed.
            try:
                actual = mut_func(*inp)
            except Exception:
                killed.add(mid)
                continue

            # Behavioral difference on this input -> the test observes the
            # mutation -> mutant is killed by this test.
            if actual != expected:
                killed.add(mid)

        kill_matrix[tid] = killed
    return kill_matrix

kill_matrix = build_kill_matrix(test_suite, mutants)
total_kills = sum(len(s) for s in kill_matrix.values())
print(f"Kill matrix shape: {len(kill_matrix)} tests x {len(mutants)} mutants")
print(f"Total (test, mutant) kill entries: {total_kills}")


Kill matrix shape: 512 tests x 47 mutants
Total (test, mutant) kill entries: 6428


## Baseline Mutation Score

A mutant is killed by the suite if **any** test kills it. Mutation score = `|killed mutants| / |total mutants|`.

In [13]:
def mutation_score(kill_matrix, selected_tids, num_mutants):
    killed = set()
    for tid in selected_tids:
        killed |= kill_matrix[tid]
    return killed, len(killed) / num_mutants

baseline_killed, baseline_score = mutation_score(
    kill_matrix, list(kill_matrix.keys()), len(mutants)
)
print(f"Baseline mutation score: {baseline_score*100:.2f}% "
      f"({len(baseline_killed)}/{len(mutants)} mutants killed)")
if len(baseline_killed) < len(mutants):
    surviving = sorted(set(mutants) - baseline_killed)
    print(f"Surviving mutants (not killable by ANY test in the suite): {surviving}")

Baseline mutation score: 78.72% (37/47 mutants killed)
Surviving mutants (not killable by ANY test in the suite): [3, 4, 5, 9, 11, 14, 22, 24, 27, 39]


## Greedy Minimization

We minimize the suite while preserving the *same* mutation score (i.e., keeping every mutant that was originally killed, killed).

This is a classic **set-cover** problem: pick tests so their union of killed mutants equals the original killed set. Exact set-cover is NP-hard; the standard greedy heuristic gives an `O(log n)`-approximation.

**TODO**: Implement `greedy_minimize(kill_matrix, target_mutants)`:
- Repeatedly pick the test that kills the most still-uncovered target mutants.
- Stop when every target mutant is covered (or no test contributes anything new).

In [16]:
def greedy_minimize(kill_matrix, target_mutants, test_suite=None, verbose=True):
    """Greedy set-cover over the kill matrix.

    At each step, pick the test that kills the most still-uncovered target
    mutants. Stop when every target mutant is covered, or no remaining test
    contributes anything new.

    Set `verbose=True` (default) to print a one-line summary at every pick:
    the chosen test's id, how many *new* mutants it kills, which ones, and
    how many target mutants remain to cover. This makes the iterative
    set-cover behavior visible.
    """
    remaining = set(target_mutants)
    selected = []
    candidates = set(kill_matrix.keys())
    target_total = len(remaining)

    if verbose:
        print(f"Starting greedy minimization: need to cover {target_total} mutants "
              f"with {len(candidates)} candidate tests")
        print("-" * 100)

    step = 0
    while remaining:
        step += 1
        best_tid, best_count = None, 0
        for tid in candidates:
            count = len(kill_matrix[tid] & remaining)
            if count > best_count:
                best_tid, best_count = tid, count
        if best_tid is None:
            if verbose:
                print(f"  [step {step}] no remaining test can kill any of the "
                      f"{len(remaining)} still-uncovered mutants -> stop")
            break

        newly_killed = sorted(kill_matrix[best_tid] & remaining)
        selected.append(best_tid)
        candidates.discard(best_tid)
        remaining -= kill_matrix[best_tid]

        if verbose:
            covered = target_total - len(remaining)
            inp_str = ""
            if test_suite is not None:
                inp, expected = test_suite[best_tid]
                inp_str = f"  input={inp} -> expected={expected}"
            print(f"  [step {step}] picked t#{best_tid:>3}  "
                  f"+{best_count:>2} new kills (mutants {newly_killed})  "
                  f"|  covered {covered}/{target_total}  remaining {len(remaining)}"
                  f"{inp_str}")

    if verbose:
        print("-" * 100)
        print(f"Done. Selected {len(selected)} tests covering {target_total - len(remaining)}/{target_total} target mutants.")
    return selected

minimized_tids = greedy_minimize(kill_matrix, baseline_killed, test_suite=test_suite)
minimized_killed, minimized_score = mutation_score(
    kill_matrix, minimized_tids, len(mutants)
)
print()
print(f"Selected {len(minimized_tids)} tests")
print(f"Minimized mutation score: {minimized_score*100:.2f}% "
      f"({len(minimized_killed)}/{len(mutants)} mutants killed)")
assert minimized_killed == baseline_killed, "Minimization must preserve mutation score!"
print("Mutation score preserved.")


Starting greedy minimization: need to cover 37 mutants with 512 candidate tests
----------------------------------------------------------------------------------------------------
  [step 1] picked t# 82  +21 new kills (mutants [1, 2, 6, 7, 13, 15, 16, 17, 26, 28, 29, 30, 34, 35, 36, 37, 42, 43, 44, 45, 46])  |  covered 21/37  remaining 16  input=(10000, 20.0, 50, True, 'NONE', 'NEW_USER') -> expected=8000
  [step 2] picked t# 40  + 7 new kills (mutants [8, 10, 21, 23, 38, 40, 41])  |  covered 28/37  remaining 9  input=(10000, 5.0, 10, True, 'WOW', 'NEW_USER') -> expected=1500
  [step 3] picked t#122  + 6 new kills (mutants [18, 19, 20, 31, 32, 33])  |  covered 34/37  remaining 3  input=(10000, 25.0, 100, True, 'NONE', 'NEW_USER') -> expected=13000
  [step 4] picked t#263  + 3 new kills (mutants [12, 25, 47])  |  covered 37/37  remaining 0  input=(45000, 3.0, 5, False, 'NONE', 'NONE') -> expected=0
---------------------------------------------------------------------------------------

## Results

In [15]:
from tabulate import tabulate

rows = [
    ["Number of tests",         len(test_suite),                 len(minimized_tids)],
    ["Mutants killed",          f"{len(baseline_killed)}/{len(mutants)}",
                                f"{len(minimized_killed)}/{len(mutants)}"],
    ["Mutation score",          f"{baseline_score*100:.2f}%",
                                f"{minimized_score*100:.2f}%"],
]
print(tabulate(rows, headers=["", "Original suite", "Minimized suite"], tablefmt="github"))
print()
print(f"Size reduction: {(1 - len(minimized_tids)/len(test_suite))*100:.1f}%")
print()
print("Selected test inputs:")
for tid in minimized_tids:
    inp, expected = test_suite[tid]
    print(f"  t#{tid:>3}: {inp}  -> {expected}")

|                 | Original suite   | Minimized suite   |
|-----------------|------------------|-------------------|
| Number of tests | 512              | 4                 |
| Mutants killed  | 37/47            | 37/47             |
| Mutation score  | 78.72%           | 78.72%            |

Size reduction: 99.2%

Selected test inputs:
  t# 82: (10000, 20.0, 50, True, 'NONE', 'NEW_USER')  -> 8000
  t# 40: (10000, 5.0, 10, True, 'WOW', 'NEW_USER')  -> 1500
  t#122: (10000, 25.0, 100, True, 'NONE', 'NEW_USER')  -> 13000
  t#263: (45000, 3.0, 5, False, 'NONE', 'NONE')  -> 0


## Reflection

- Greedy is fast but **not optimal**. Try a brute-force search over small subsets and see if you can find a smaller cover.
- The minimized suite **preserves mutation score**, but mutation score is itself only a proxy for real fault detection. A minimized suite may miss faults that the original suite would have detected. What does this say about using minimization in safety-critical contexts?
- What changes if you switch the *sufficiency criterion* from mutation kills to statement coverage? Which redundancies disappear, and which appear?